# 🌌 microWakeWord Training: "nebula" (Cleaned & Optimized)

This notebook trains an **on-device wake word model** for **"nebula"** using the **microWakeWord** framework, optimized for the ESP32-S3.

### 📁 File Structure Requirements:
This notebook expects a specific folder structure in your Google Drive. **Step 0** will automatically create this structure for you.

* `SIH_KWS/positives/`: Your wake word audio clips (1-2 seconds, `.wav` or `.mp3`).
* `SIH_KWS/hard_negatives/`: Words that sound similar to your wake word (e.g., "Nebuchadnezzar", "fabulous").
* `SIH_KWS/augmentation/musan_subset/noise/`: (Optional) Custom background noise clips for data augmentation.
* `SIH_KWS/trained_models/`: The final model, C-headers, and ESPHome JSON will be exported here automatically.

### ⚠️ Setup Reminder:
Ensure your Colab runtime is set to use a GPU (**Runtime > Change runtime type > T4 GPU**).

## Step 0: Mount Google Drive & Initialize Folder Structure
Run this cell first. It will connect to your Google Drive and build the exact folder structure the training pipeline needs.

**Instructions:** Run this cell, upload your audio files to the newly created folders in your Google Drive, and then continue to Step 1.

In [ ]:
import os
from google.colab import drive

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the exact paths needed for the pipeline
base_path = "/content/drive/MyDrive/SIH_KWS"

directories = [
    f"{base_path}/positives",
    f"{base_path}/hard_negatives",
    f"{base_path}/augmentation/musan_subset/noise",
    f"{base_path}/trained_models"
]

print("📁 Initializing Google Drive Folder Structure...\n")

# 3. Create the directories if they don't exist
for directory in directories:
    os.makedirs(directory, exist_ok=True)
    print(f"✅ Verified: {directory}")

print("\n🛑 ACTION REQUIRED 🛑")
print("If you haven't already, please upload your audio files to the 'positives' and 'hard_negatives' folders in your Google Drive.")
print("Once your files are uploaded, proceed to Step 1.")

## Step 1: Configure Paths & Verify Data
We define the paths to your dataset and create an auto-versioned output directory to ensure older models are not overwritten.

In [ ]:
import os

DRIVE_POSITIVES_PATH = "/content/drive/MyDrive/SIH_KWS/positives"
DRIVE_HARD_NEGATIVES_PATH = "/content/drive/MyDrive/SIH_KWS/hard_negatives"
base_output_dir = "/content/drive/MyDrive/SIH_KWS/trained_models"

model_folder = "nebula"
target_dir = os.path.join(base_output_dir, model_folder)

# Auto-versioning: Check if folder exists and increment (_v1, _v2, etc.)
if os.path.exists(target_dir):
    version = 1
    while os.path.exists(f"{target_dir}_v{version}") or os.path.exists(f"{target_dir}-v{version}"):
        version += 1
    GOOGLE_DRIVE_OUTPUT_DIR = f"{target_dir}_v{version}"
else:
    GOOGLE_DRIVE_OUTPUT_DIR = target_dir

os.makedirs(GOOGLE_DRIVE_OUTPUT_DIR, exist_ok=True)

# Verify data exists before proceeding
pos_files = os.listdir(DRIVE_POSITIVES_PATH) if os.path.exists(DRIVE_POSITIVES_PATH) else []
neg_files = os.listdir(DRIVE_HARD_NEGATIVES_PATH) if os.path.exists(DRIVE_HARD_NEGATIVES_PATH) else []

print("\n📁 Paths Configured:")
print(f" - Positives:      {DRIVE_POSITIVES_PATH} -> {'✅ FOUND' if len(pos_files) > 0 else '❌ FOLDER EMPTY'}")
print(f" - Hard Negatives: {DRIVE_HARD_NEGATIVES_PATH} -> {'✅ FOUND' if len(neg_files) > 0 else '⚠️ WARNING: EMPTY (Highly recommended to add hard negatives)'}")
print(f" - Output folder:  {GOOGLE_DRIVE_OUTPUT_DIR}")

if len(pos_files) == 0:
    raise FileNotFoundError("Positives folder is empty! Please upload wake word samples before continuing.")

## Step 2: Install Dependencies & Patch microWakeWord
Installs the required audio processing libraries, clones the `micro-wake-word` repository, and automatically applies a known bug fix (`ProcessSamples`) to the framework.

In [ ]:
import os

# Install required packages
!pip install -q audiomentations mmap_ninja datasets pyyaml webrtcvad-wheels ai-edge-litert scipy tqdm librosa soundfile
!pip install -q 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
!pip install -q 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# Clone microWakeWord from OHF-Voice
if not os.path.exists('micro-wake-word'):
    !git clone https://github.com/OHF-Voice/micro-wake-word.git
    !pip install -e ./micro-wake-word

# Patch audio_utils.py to fix the ProcessSamples casing issue
audio_utils_path = "/content/micro-wake-word/microwakeword/audio/audio_utils.py"
with open(audio_utils_path, 'r') as f:
    content = f.read()
patched = content.replace('micro_frontend.process_samples(', 'micro_frontend.ProcessSamples(')
with open(audio_utils_path, 'w') as f:
    f.write(patched)

print("✅ Packages installed and framework patched successfully!")

## Step 3: Ingest & Normalize Audio to Local SSD
Training directly from Google Drive is slow. We copy all audio to the local Colab disk, converting them to the required format (16 kHz, 16-bit PCM, Mono WAV) while trimming leading/trailing silence.

In [ ]:
import os
from pathlib import Path
import soundfile as sf
import librosa
from tqdm import tqdm

local_pos_dir = '/content/my_data/positive'
local_neg_dir = '/content/my_data/hard_negatives'
os.makedirs(local_pos_dir, exist_ok=True)
os.makedirs(local_neg_dir, exist_ok=True)

def normalize_folder(src_path, dest_dir, label):
    files = [f for f in Path(src_path).rglob('*') if f.suffix.lower() in ['.wav', '.mp3', '.flac', '.ogg', '.m4a']]
    count = 0
    if not files:
        return count
    for f in tqdm(files, desc=f"Normalizing {label}"):
        try:
            y, sr = librosa.load(str(f), sr=16000, mono=True)
            y_trimmed, _ = librosa.effects.trim(y, top_db=30)
            out_name = f"{count}_{f.stem}.wav"
            sf.write(os.path.join(dest_dir, out_name), y_trimmed, 16000, subtype='PCM_16')
            count += 1
        except Exception:
            pass
    return count

print("Converting and copying audio to local SSD...")
pos_count = normalize_folder(DRIVE_POSITIVES_PATH, local_pos_dir, "Positives")
neg_count = normalize_folder(DRIVE_HARD_NEGATIVES_PATH, local_neg_dir, "Hard Negatives")

print(f"\n📊 Local SSD Dataset Ready:")
print(f" - Positives:      {pos_count} files in {local_pos_dir}")
print(f" - Hard Negatives: {neg_count} files in {local_neg_dir}")

## Step 4: Download Background Audio & Negative Spectrograms
To make the model resilient in real-world environments, we download MIT Room Impulse Responses (reverb), AudioSet, and pre-computed conversational noise from Hugging Face.

In [ ]:
import os
import shutil
from pathlib import Path
from tqdm import tqdm
from huggingface_hub import snapshot_download
import librosa
import soundfile as sf

# 1. Download Room Impulse Responses (MIT RIRs)
rir_dir = "/content/mit_rirs"
os.makedirs(rir_dir, exist_ok=True)
if len([f for f in os.listdir(rir_dir) if f.endswith('.wav')]) < 50:
    print("Downloading MIT RIR dataset...")
    raw_dir = "/content/mit_rirs_raw"
    snapshot_download(
        repo_id="davidscripka/MIT_environmental_impulse_responses",
        repo_type="dataset",
        local_dir=raw_dir,
        allow_patterns=["16khz/*.wav", "*.wav"],
    )
    for f in Path(raw_dir).rglob("*.wav"):
        shutil.copy(str(f), os.path.join(rir_dir, f.name))
    shutil.rmtree(raw_dir, ignore_errors=True)

# 2. Download Background Noise (AudioSet sample clips)
os.makedirs('/content/audioset_16k', exist_ok=True)
if len([f for f in os.listdir('/content/audioset_16k') if f.endswith('.wav')]) < 50:
    print("Downloading AudioSet sample background audio...")
    !wget -q -O /content/audioset_sample.tar https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar
    !mkdir -p /content/audioset_raw && tar -xf /content/audioset_sample.tar -C /content/audioset_raw
    for f in tqdm(list(Path('/content/audioset_raw').glob('**/*.flac'))[:200], desc="Converting AudioSet"):
        try:
            y, sr = librosa.load(str(f), sr=16000, mono=True)
            sf.write(os.path.join('/content/audioset_16k', f"{f.stem}.wav"), y, 16000, subtype='PCM_16')
        except Exception:
            pass
    shutil.rmtree('/content/audioset_raw', ignore_errors=True)

# 3. Download Hugging Face Pre-Generated Negative Spectrograms
os.makedirs('/content/negative_datasets', exist_ok=True)
hf_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
neg_zips = ['dinner_party.zip', 'dinner_party_eval.zip', 'no_speech.zip', 'speech.zip']

for fname in neg_zips:
    target_dir = os.path.join('/content/negative_datasets', fname.replace('.zip', ''))
    if not os.path.exists(target_dir):
        print(f"Downloading {fname} from Hugging Face...")
        zip_file = f"/content/negative_datasets/{fname}"
        !wget -q -O {zip_file} {hf_root}{fname}
        !unzip -q -o {zip_file} -d /content/negative_datasets
        os.remove(zip_file)

print("\n✅ Background audio and negative features downloaded successfully!")

## Step 5: Feature Augmentation & Spectrogram Generation
We augment the local audio clips (pitch shifts, EQ, noise injection) and convert them into memory-mapped 40-band spectrograms for ultra-fast training.

In [ ]:
import sys
sys.path.insert(0, '/content/micro-wake-word')

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
from pathlib import Path
import shutil, os, random

local_noise_dir = "/content/background_noise"
os.makedirs(local_noise_dir, exist_ok=True)
musan_noise_source = Path("/content/drive/MyDrive/SIH_KWS/augmentation/musan_subset/noise")

# Safely check if custom noise was uploaded
noise_files = list(musan_noise_source.rglob("*.wav")) + list(musan_noise_source.rglob("*.mp3"))
if noise_files:
    selected_noise = random.sample(noise_files, min(200, len(noise_files)))
    print(f"Staging {len(selected_noise)} custom noise clips to local SSD...")
    for f in selected_noise:
        shutil.copy(str(f), os.path.join(local_noise_dir, f.name))
else:
    print("⚠️ No custom noise found in Drive. Proceeding with AudioSet and downloaded backgrounds only.")

# Configure Augmentation Pipeline
augmenter = Augmentation(
    augmentation_duration_s=3.0,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.2,
        "TanhDistortion": 0.15,
        "PitchShift": 0.25,
        "BandStopFilter": 0.15,
        "AddColorNoise": 0.2,
        "AddBackgroundNoise": 0.80,
        "Gain": 1.0,
        "RIR": 0.60,
    },
    impulse_paths=['/content/mit_rirs'],
    background_paths=[local_noise_dir, '/content/audioset_16k'],
    background_min_snr_db=-5,
    background_max_snr_db=15,
    min_jitter_s=0.15,
    max_jitter_s=0.25,
)

def process_dataset(input_dir, output_features_dir, repeat_train=3):
    if not os.path.exists(input_dir) or not os.listdir(input_dir):
        print(f"Skipping {input_dir} (Folder is empty)")
        return
        
    print(f"\nProcessing {input_dir} -> {output_features_dir}...")
    if os.path.exists(output_features_dir):
        shutil.rmtree(output_features_dir)
    os.makedirs(output_features_dir, exist_ok=True)

    clips = Clips(
        input_directory=input_dir,
        file_pattern='*.wav',
        max_clip_duration_s=None,
        remove_silence=False,
        random_split_seed=42,
        split_count=0.1,
    )

    for split in ["training", "validation", "testing"]:
        out_dir = os.path.join(output_features_dir, split)
        os.makedirs(out_dir, exist_ok=True)

        if split == "training":
            split_name, rep, slide_frames = "train", repeat_train, 10
        elif split == "validation":
            split_name, rep, slide_frames = "validation", 1, 10
        else:
            split_name, rep, slide_frames = "test", 1, 1

        spectrograms = SpectrogramGeneration(clips=clips, augmenter=augmenter, slide_frames=slide_frames, step_ms=10)
        RaggedMmap.from_generator(
            out_dir=os.path.join(out_dir, 'wakeword_mmap'),
            sample_generator=spectrograms.spectrogram_generator(split=split_name, repeat=rep),
            batch_size=100,
            verbose=False,
        )
        print(f"  ✅ Generated {split} features")

process_dataset('/content/my_data/positive', '/content/features/positive', repeat_train=3)
process_dataset('/content/my_data/hard_negatives', '/content/features/hard_negatives', repeat_train=2)
print("\n✅ All feature maps successfully generated!")

## Step 6: Configure Training Parameters
We configure the neural network parameters, applying heavy penalties to hard negatives to prevent false triggers, and combining all datasets for robust training.

In [ ]:
import yaml, shutil, os

config = {
    "window_step_ms": 10,
    "train_dir": "trained_models/nebula",
    "features": [
        {
            "features_dir": "/content/features/positive",
            "sampling_weight": 4.0,
            "penalty_weight": 1.0,
            "truth": True,
            "truncation_strategy": "truncate_start",
            "type": "mmap",
        },
        {
            "features_dir": "/content/negative_datasets/speech",
            "sampling_weight": 10.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "/content/negative_datasets/dinner_party",
            "sampling_weight": 10.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
        {
            "features_dir": "/content/negative_datasets/no_speech",
            "sampling_weight": 5.0,
            "penalty_weight": 1.0,
            "truth": False,
            "truncation_strategy": "random",
            "type": "mmap",
        },
    ],
    "training_steps": [10000],
    "positive_class_weight": [1],
    "negative_class_weight": [25],
    "learning_rates": [0.001],
    "batch_size": 64,
    "eval_step_interval": 2500,
    "clip_duration_ms": 1500,
    "time_mask_max_size": [0],
    "time_mask_count": [0],
    "freq_mask_max_size": [0],
    "freq_mask_count": [0],
    "target_minimization": 0.9,
    "minimization_metric": None,
    "maximization_metric": "average_viable_recall",
}

# Add hard negatives dynamically if they were generated
if os.path.exists("/content/features/hard_negatives"):
    config["features"].insert(1, {
        "features_dir": "/content/features/hard_negatives",
        "sampling_weight": 4.0,
        "penalty_weight": 2.5,
        "truth": False,
        "truncation_strategy": "truncate_start",
        "type": "mmap",
    })

# Remove old corrupted checkpoints
if os.path.exists("trained_models/nebula"):
    shutil.rmtree("trained_models/nebula")

with open("training_parameters.yaml", "w") as f:
    yaml.dump(config, f, sort_keys=False)

print("✅ Configuration updated successfully!")

## Step 7: Train Model & Export Quantized Streaming TFLite
Executes the training loop using the `mixednet` architecture. The model is then tested and quantized to INT8, optimized for ESP32.

In [ ]:
import sys
sys.path.insert(0, '/content/micro-wake-word')

!python -m microwakeword.model_train_eval \
  --training_config='training_parameters.yaml' \
  --train 1 \
  --restore_checkpoint 1 \
  --test_tf_nonstreaming 0 \
  --test_tflite_nonstreaming 0 \
  --test_tflite_nonstreaming_quantized 0 \
  --test_tflite_streaming 0 \
  --test_tflite_streaming_quantized 1 \
  --use_weights "best_weights" \
  mixednet \
  --pointwise_filters "64,64,64,64" \
  --repeat_in_block "1, 1, 1, 1" \
  --mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' \
  --residual_connection "0,0,0,0" \
  --first_conv_filters 32 \
  --first_conv_kernel_size 5 \
  --stride 3

## Step 8: View ROC Test Results
Reviews the False Accepts per Hour (`faph`) vs False Rejection Rate (`frr`). Pick a cutoff where false accepts are zero.

In [ ]:
roc_file = "trained_models/nebula/tflite_stream_state_internal_quant/tflite_streaming_roc.txt"
if os.path.exists(roc_file):
    print("--- ROC Results ---")
    with open(roc_file, 'r') as f:
        print(f.read())
    print("\n💡 Recommendation: Pick a probability cutoff where False Accepts/hr is near 0 (typically 0.78 - 0.88).")
else:
    print(f"ROC file {roc_file} not found yet.")

## Step 9: Export C Header & Save Directly to Google Drive
Converts the INT8 TFLite model into a C byte array (`nebula_model.h`) and exports all assets back to your Google Drive automatically.

In [ ]:
from google.colab import files
import shutil
import os

# 0. Force create the Google Drive directory
os.makedirs(GOOGLE_DRIVE_OUTPUT_DIR, exist_ok=True)

tflite_path = "trained_models/nebula/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
header_path = "nebula_model.h"
manifest_path = "nebula.json"
roc_file = "trained_models/nebula/tflite_stream_state_internal_quant/tflite_streaming_roc.txt"

# 1. Generate C Header for ESP-IDF / Arduino (Pure Python)
if os.path.exists(tflite_path):
    with open(tflite_path, 'rb') as f:
        tflite_data = f.read()

    hex_array = ', '.join([f'0x{b:02x}' for b in tflite_data])
    c_code = f"const unsigned char nebula_model_tflite[] = {{{hex_array}}};\n"
    c_code += f"const unsigned int nebula_model_tflite_len = {len(tflite_data)};\n"

    with open(header_path, 'w') as f:
        f.write(c_code)
    print(f"✅ Generated C Header: {header_path}")
else:
    print(f"❌ Error: Model file not found at {tflite_path}")

# 2. Generate ESPHome Manifest
manifest_content = '''{
  "version": 2,
  "model_type": "mixednet",
  "wake_word": "nebula",
  "probability_cutoff": 0.82,
  "sliding_window_size": 3,
  "author": "Custom microWakeWord Model",
  "tensor_arena_size": 120000
}'''
with open(manifest_path, 'w') as f:
    f.write(manifest_content)
print(f"✅ Generated ESPHome Manifest: {manifest_path}")

# 3. Save directly back to your Google Drive folder
drive_tflite = os.path.join(GOOGLE_DRIVE_OUTPUT_DIR, "stream_state_internal_quant.tflite")
drive_header = os.path.join(GOOGLE_DRIVE_OUTPUT_DIR, "nebula_model.h")
drive_manifest = os.path.join(GOOGLE_DRIVE_OUTPUT_DIR, "nebula.json")
drive_roc = os.path.join(GOOGLE_DRIVE_OUTPUT_DIR, "tflite_streaming_roc.txt")

if os.path.exists(tflite_path):
    shutil.copy(tflite_path, drive_tflite)
if os.path.exists(header_path):
    shutil.copy(header_path, drive_header)
if os.path.exists(manifest_path):
    shutil.copy(manifest_path, drive_manifest)
if os.path.exists(roc_file):
    shutil.copy(roc_file, drive_roc)

print(f"\n💾 All artifacts successfully saved to Google Drive at:\n   {GOOGLE_DRIVE_OUTPUT_DIR}")

# 4. Provide direct browser download bundle
!zip -q -j nebula_esp32_deployment.zip {tflite_path} {header_path} {manifest_path}
files.download("nebula_esp32_deployment.zip")
print("✅ Download triggered!")